In [13]:
from ccka.models.kernel import KernelModel, HardwareKernelRunner, HardwareKernelModel
from ccka.circuits.angleEmbeddingKernel import QuackEmbeddingQiskitCircuit
from ccka.aligner.kta import fullKTA, centroidBasedKTA, quackKTA, randomKTA, greedyKTA
import pennylane as qml
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
import os
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [14]:
def _make_circular_data(num_sectors):
    """Generate datapoints arranged in an even circle."""
    center_indices = np.array(range(0, num_sectors))
    sector_angle = 2 * np.pi / num_sectors
    angles = (center_indices + 0.5) * sector_angle
    x = 0.7 * np.cos(angles)
    y = 0.7 * np.sin(angles)
    labels = 2 * np.remainder(np.floor_divide(angles, sector_angle), 2) - 1

    return x, y, labels


def make_double_cake_data(num_sectors):
    x1, y1, labels1 = _make_circular_data(num_sectors)
    x2, y2, labels2 = _make_circular_data(num_sectors)

    # x and y coordinates of the datapoints
    x = np.hstack([x1, 0.5 * x2])
    y = np.hstack([y1, 0.5 * y2])

    # Canonical form of dataset
    X = np.vstack([x, y]).T

    labels = np.hstack([labels1, -1 * labels2])

    # Canonical form of labels
    Y = labels.astype(int)

    return X, Y

X, y = make_double_cake_data(num_sectors=4)
X.shape, y.shape

((8, 2), (8,))

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# First time only — saves credentials to disk
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token= '2_B2bfHWRFAQzAN72FxCcy6KMuvmIzysyM0vK4iuhvFq', #'TzFqvDVsCLogQCHMC8uMFVDqoXVesYpXef9i-48OCade',
    overwrite=True,
)

# Load saved account
service = QiskitRuntimeService()

# List available backends and pick the least busy with enough qubits
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=4,          # match your num_qubits
)
print(f"Using backend: {backend.name}")

# ── Circuit and runner ─────────────────────────────────────────────────
num_qubits = 2
reps       = 1

circuit = QuackEmbeddingQiskitCircuit(num_qubits=num_qubits, reps=reps, reupload=True)
runner  = HardwareKernelRunner(
    circuit=circuit,
    backend=backend,
    shots=1024,
    batch_size=1,          # tune to your backend's limits
    optimization_level=3,
    mitigation_level=0,
)

kernel_model = HardwareKernelModel(circuit=circuit, runner=runner)

InvalidAccountError: 'Unable to retrieve instances. Please check that you are using a valid API token.'

In [ ]:
# ── Centroid-Based KTA (recommended for hardware — fewest kernel evaluations) ────
aligner = centroidBasedKTA(
                    kernel_model= kernel_model,
                    data = X,
                    labels = y,
                    matrix_type='regular',
                    clustering='regular',
                    split_size=0.50,
                    centroids= 2,
                    lambda_co=0.0,
                    lambda_kao=0.0,
                    epochs=10,
                    eps=0.001,
                    alpha=0.01
        )

history = aligner.align()
optimized_weights = history["weights"]
print(f"Best test accuracy : {history['best_test_accuracy']:.4f}")
print(f"Circuit executions : {history['circuit_executions']}")

Submitting batches: 100%|██████████| 10/10 [00:14<00:00,  1.44s/it]



All 10 job(s) submitted. Waiting for results...


Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))


Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))


Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))


...

Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))


Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))


Data: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=2>))



[CentroidBasedKTA] KTA alignment:   3%|▎         | 1/30 [28:23<13:43:31, 1703.85s/it]


KeyboardInterrupt: 